<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-5-rag-pipeline/week5_RAG_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Assignment | RAG Pipeline with Retrieval Evaluation
## COSC 650 Applied LLM Systems
###Ben Miller

# Part 1: Build the pipeline
Ingest and chunk a documentation corpus of at least eight documents, embed the chunks locally with sentence-transformers, and store them in a vector index (FAISS or similar). Implement retrieval that embeds a query and returns the top-k chunks.

### Download documents from my repo

In [1]:
import os, re, pathlib, numpy as np


BRANCH = "week-5-rag-pipeline"
REPO_URL = "https://github.com/BenMillerDev/Applied-LLM-Systems.git"

if not os.path.exists('docs'):
    # docs/ isn't sitting next to this notebook already (e.g. it was opened
    # on its own rather than as part of a full repo clone) -- pull it down
    !rm -rf _repo
    !git clone -b {BRANCH} {REPO_URL} _repo
    !cp -r _repo/week-5/docs .

import pathlib

TOP_K = 3
DOCS_DIR = pathlib.Path('docs')
DOCS = {p.stem: p.read_text() for p in sorted(DOCS_DIR.glob('*.md'))}

print(f'loaded {len(DOCS)} docs:', list(DOCS.keys()))
print(f'total chars: {sum(len(v) for v in DOCS.values())}')
print()
print('--- architecture, first 400 chars ---')
print(DOCS['architecture'][:400])

loaded 11 docs: ['analytics', 'architecture', 'auth', 'availability', 'client_profiles', 'customer_booking_flow', 'onboarding', 'realtime_sync', 'roadmap', 'service_management', 'time_slot_algorithm']
total chars: 12113

--- architecture, first 400 chars ---
## NailSalonApp Architecture

NailSalonApp is a full-stack booking platform for independent nail technicians, split into two separate codebases that serve two different audiences. The owner-facing app is a native iOS app built with React Native and Expo, written in TypeScript, and used daily to run the business. Customers never install anything -- they book through a separate React (TypeScript) we


### Set up the SentenceTransformer embedder

In [2]:
from sentence_transformers import SentenceTransformer

!pip install -q faiss-cpu sentence-transformers
import faiss

os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Chunking functions

In [3]:
def chunk_fixed(text, chunk_size, overlap):
    """Split text into fixed-size, overlapping chunks (used for the 'small' and 'large' configs)."""
    normalized_text = re.sub(r'\s+', ' ', text).strip() # Collapse newlines and repeated spaces into single spaces
    chunks = []
    start = 0
    while start < len(normalized_text):
        end = start + chunk_size
        chunks.append(normalized_text[start:end])
        start += chunk_size - overlap  # step forward by less than chunk_size so chunks overlap
    return chunks

def small_chunker(text):
    """Chunk text into small, overlapping pieces (120 characters, 20-character overlap)."""
    return chunk_fixed(text, 120, 20)

def large_chunker(text):
    """Chunk text into larger, overlapping pieces (320 characters, 40-character overlap)."""
    return chunk_fixed(text, 320, 40)

def chunk_paragraph(text):
    """Split text into one chunk per paragraph (blank-line-separated)."""
    raw_paragraphs = text.split('\n\n')
    paragraphs = []
    for paragraph in raw_paragraphs:
        cleaned_paragraph = re.sub(r'\s+', ' ', paragraph).strip() # Collapse newlines and repeated spaces into single spaces
        if cleaned_paragraph:  # skip empty paragraphs, e.g. a leading or trailing blank line
            paragraphs.append(cleaned_paragraph)
    return paragraphs

def build_chunks(docs, chunk_fn):
    """
    Chunk each document separately (not one concatenated blob) so a chunk
    never straddles two different docs, and tag each chunk with the doc
    name it came from -- Part 2's relevance labeling needs that source tag.
    """
    all_chunks = [] # every chunk from every doc, flattened into one list
    chunk_sources = [] # same length as all_chunks, chunk_sources[i] names the doc all_chunks[i] came from
    for doc_name, doc_text in docs.items():
        doc_chunks = chunk_fn(doc_text)
        all_chunks.extend(doc_chunks)
        chunk_sources.extend([doc_name] * len(doc_chunks))
    return all_chunks, chunk_sources

### Compare three chunking configurations: small (120 characters), large (320 characters), and by paragraph

In [4]:
CHUNK_CONFIGS = {
    'small (120/20)': small_chunker,
    'large (320/40)': large_chunker,
    'by-paragraph': chunk_paragraph,
}

configs = {}
for cfg_name, chunk_fn in CHUNK_CONFIGS.items():
    chunks, sources = build_chunks(DOCS, chunk_fn)
    configs[cfg_name] = {'chunks': chunks, 'sources': sources}
    print(f'{cfg_name:16s} -> {len(chunks)} chunks')

small (120/20)   -> 127 chunks
large (320/40)   -> 47 chunks
by-paragraph     -> 42 chunks


### Embed the chunks with SentenceTransformers

In [5]:
def build(chunks):
    """Embed a list of chunks and load them into a FAISS index for similarity search."""
    chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings)
    return index

indices = {name: build(cfg['chunks']) for name, cfg in configs.items()}

### Chunk Retrieval functions

In [6]:
def retrieve(index, chunks, query, k=3):
    """Return the actual chunk text for the top-k chunks most similar to the query."""
    top_positions = retrieve_indices(index, query, k)
    return [chunks[position] for position in top_positions]

def retrieve_indices(index, query, k):
    """Return the positions of the top-k chunks most similar to the query."""
    query_embedding = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, retrieved_positions = index.search(query_embedding, k)
    # FAISS pads with -1 when fewer than k chunks exist in the index -- drop those.
    return [position for position in retrieved_positions[0] if position >= 0]

### Evaluation functions for Part 2

In [14]:
def label_relevant(chunks, sources, source_doc, fact):
    """
    Find every chunk that counts as ground-truth relevant for one query:
    it has to come from the right document AND actually contain the
    fact's wording -- checked per config, since each chunking config
    slices the same docs into a different set of chunk strings.
    """
    fact_lower = fact.lower()
    relevant_chunk_indices = []
    for i, (chunk_text, chunk_source) in enumerate(zip(chunks, sources)):
        is_from_right_doc = chunk_source == source_doc
        contains_the_fact = fact_lower in chunk_text.lower()
        if is_from_right_doc and contains_the_fact:
            relevant_chunk_indices.append(i)
    return relevant_chunk_indices

def precision_recall_at_k(index, chunks, sources, query, source_doc, fact, k=TOP_K):
    """
    Score one query against one chunking config: precision is how many of
    the k retrieved chunks were actually relevant; recall is how many of
    the relevant chunks that exist were actually retrieved.
    """
    retrieved_chunk_indices = retrieve_indices(index, query, k)
    relevant_chunk_indices = label_relevant(chunks, sources, source_doc, fact)

    relevant_and_retrieved = []
    for chunk_index in retrieved_chunk_indices:
        if chunk_index in relevant_chunk_indices:
            relevant_and_retrieved.append(chunk_index)

    precision = len(relevant_and_retrieved) / len(retrieved_chunk_indices) if retrieved_chunk_indices else 0.0
    recall = len(relevant_and_retrieved) / len(relevant_chunk_indices) if relevant_chunk_indices else 0.0
    return precision, recall

def show_retrieved(cfg_name, cfg, index, query, source_doc, fact, k=TOP_K):
    """
    Print what a config's index actually returned for one query -- precision/
    recall plus the top-k chunks themselves, each flagged relevant/wrong and
    tagged with its source doc, so it's clear *what* got mixed up instead of
    just *that* something did.
    """
    relevant_chunk_indices = label_relevant(cfg['chunks'], cfg['sources'], source_doc, fact)
    retrieved_chunk_indices = retrieve_indices(index, query, k)
    precision, recall = precision_recall_at_k(index, cfg['chunks'], cfg['sources'], query, source_doc, fact, k)

    print(f'    {cfg_name:16s} precision={precision:.2f} recall={recall:.2f}')
    for chunk_index in retrieved_chunk_indices:
        flag = 'relevant' if chunk_index in relevant_chunk_indices else 'wrong'
        source = cfg['sources'][chunk_index]
        snippet = cfg['chunks'][chunk_index][:90].replace('\n', ' ')
        print(f'        [{flag:8s}] ({source:20s}) "{snippet}..."')

def evaluate_all_configs(configs, indices, qa):
    """
    Score every query in `qa` against every chunking config, printing each
    config's mean precision/recall as it finishes, and return the individual
    scores keyed by query so the disagreement check below can compare across
    configs rather than just seeing the averages.
    """
    per_query_results = {}
    for cfg_name, cfg in configs.items():
        config_precisions = []
        config_recalls = []
        for query, fact, source_doc in qa:
            precision, recall = precision_recall_at_k(
                indices[cfg_name], cfg['chunks'], cfg['sources'], query, source_doc, fact
            )
            config_precisions.append(precision)
            config_recalls.append(recall)
            if query not in per_query_results:
                per_query_results[query] = {}
            per_query_results[query][cfg_name] = (precision, recall)
        print(f'{cfg_name:16s} {np.mean(config_precisions):<18.2f} {np.mean(config_recalls):<15.2f}')
    return per_query_results

# Part 2: Evaluate retrieval
Compare at least two embedding models, or at least three chunking configurations, on at least ten test queries for which you decide in advance which retrieved chunks count as relevant. Report precision and recall at the chunk level, and explain where the choices disagree and why.

In [15]:
# Query, the fact that answers it, and which doc that fact lives in
qa = [
    ('How far in advance must a customer book?', '1 hour', 'time_slot_algorithm'),
    ('What does the owner app use for navigation?', 'Expo Router', 'architecture'),
    ('How do new bookings appear on the dashboard without a refresh?', 'no pull-to-refresh or manual reload needed', 'realtime_sync'),
    ("Where do new owners' service lists start from?", 'default service catalog', 'service_management'),
    ('How many steps are in the customer booking flow?', 'four steps', 'customer_booking_flow'),
    ('What happens to Google sign-in when running in Expo Go?', 'mock version', 'auth'),
    ('What three time ranges can the analytics dashboard filter by?', 'This Week, This Month, and This Year', 'analytics'),
    ('What two fields track onboarding progress?', "dismissed the checklist", 'onboarding'),
    ("How is a client's fill due date determined?", 'computed fill due date', 'client_profiles'),
    ('What limits how many bookings an owner can take in a day?', 'maximum number of bookings allowed per day', 'availability'),
    ('Does the app support SMS booking confirmations today?', 'SMS booking confirmations', 'roadmap'),
]

print(f'{"config":16s} {"mean precision@"+str(TOP_K):18s} {"mean recall@"+str(TOP_K):15s}')
per_query_results = evaluate_all_configs(configs, indices, qa)

fact_and_doc_by_query = {query: (fact, source_doc) for query, fact, source_doc in qa}

print()
print('Per-query disagreement (configs whose precision or recall differ), with what was retrieved:')
for query, scores_by_config in per_query_results.items():
    distinct_scores = set(scores_by_config.values())
    configs_disagree = len(distinct_scores) > 1
    if configs_disagree:
        fact, source_doc = fact_and_doc_by_query[query]
        print(f'  "{query}"  ->  fact: "{fact}"  (doc: {source_doc})')
        for cfg_name, cfg in configs.items():
            show_retrieved(cfg_name, cfg, indices[cfg_name], query, source_doc, fact)
        print()

config           mean precision@3   mean recall@3  
small (120/20)   0.21               0.64           
large (320/40)   0.33               0.91           
by-paragraph     0.33               1.00           

Per-query disagreement (configs whose precision or recall differ), with what was actually retrieved:
  "How far in advance must a customer book?"  ->  fact: "1 hour"  (doc: time_slot_algorithm)
    small (120/20)   precision=0.33 recall=1.00
        [relevant] (time_slot_algorithm ) "ings say: a customer must book at least 1 hour in advance. Any slot that falls inside that..."
        [wrong   ] (availability        ) "rithm the next time a customer tries to book...."
        [wrong   ] (customer_booking_flow) "owner's hours. Once submitted, a confirmation screen shows the customer a full summary of ..."
    large (320/40)   precision=0.67 recall=1.00
        [relevant] (time_slot_algorithm ) "intments so nothing is scheduled with zero gap. It also removes any date the owner has m

I compared three chunking configurations -- small (120/20), large (320/40), and by-paragraph. A chunk counts as relevant for that query if it comes from the right document and contains the fact's wording.


---


| Config | Mean precision@3 | Mean recall@3 |
|---|---|---|
| small (120/20) | 0.21 | 0.64 |
| large (320/40) | 0.33 | 0.91 |
| by-paragraph | 0.33 | 1.00 |

By-paragraph recovered the correct chunk for every query. Large missed one out of eleven. Small missed four.

### Where the configs disagree, and why

Small chunking's failures come down to the same thing: at 120 characters each doc splits into so many little chunks that the one with the actual fact has to compete against other chunks from the same doc covering different details.  Some chunks are from unrelated docs that just happen to share a word with the query. `customer_booking_flow` chunks show up as the wrong answer in three of the four failures even though none of those questions are about the booking flow, just because almost every query touches "booking" somewhere and a 120-character chunk doesn't carry enough surrounding text to determine a real match.

Large chunking's one miss is similar the small's misses: two of the three chunks it retrieves for the booking-cap question are `customer_booking_flow` chunks about picking dates and times, pulled in on the same "booking" word. The third, closest chunk is actually from the right document about the right detail: it starts "...of bookings allowed per day, which caps how many appointments can be scheduled...", but it is missing "maximum number" off the front.  It doesn't count as a match against the exact fact string. Even a 320-character window doesn't fully dodge the boundary problem, but it does happen less often than it did at 120 characters.

By-paragraph wins overall because it chunks along the document's own topic boundaries instead of a fixed character count, so it mostly avoids the failure mode the others ran into. Each paragraph is naturally sized to hold a whole fact as one topically coherent unit, so precision stays mediocre (0.33 on most of these, same as large) while recall never slips.

# Part 3: Generate
Add a generation step that builds a grounded prompt from the retrieved chunks and answers with Gemini.

In [9]:
if 'google.colab' in str(get_ipython()):
    from google.colab import userdata
    try:
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass  # secret not set up in Colab yet -- falls back to [API-BLOCKED] below

def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE)

live model calls: True


In [10]:
def answer(question, config='by-paragraph', k=TOP_K):
    """
    Retrieve the top-k chunks for a question under one chunking config, build
    a prompt that only allows Gemini to answer from that retrieved context,
    and return its response (or a placeholder if no API key is set).
    """
    retrieved_chunks = retrieve(indices[config], configs[config]['chunks'], question, k)
    context_block = '\n- '.join(retrieved_chunks)
    prompt = f'Answer only from the context.\nContext:\n- {context_block}\nQuestion: {question}\nAnswer:'

    response_text = gemini_chat([{'role': 'user', 'content': prompt}])
    if response_text is None:
        return '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'
    return response_text

def print_case(label, question, config):
    """Print one retrieval-quality case -- its query and config -- followed by the generated answer."""
    print(f'{label}:', question, '|', config)
    print(answer(question, config=config))

print_case('HIGH-RECALL case', 'How far in advance must a customer book?', 'small (120/20)')
print()
print_case('LOW-RECALL case', 'What three time ranges can the analytics dashboard filter by?', 'small (120/20)')

HIGH-RECALL case: How far in advance must a customer book? | small (120/20)
At least 1 hour in advance.

LOW-RECALL case: What three time ranges can the analytics dashboard filter by? | small (120/20)
Based on the provided context, there is no mention of the three time ranges that the analytics dashboard can filter by.


# Part 4: Find one failure and explain it
Show a query where retrieval surfaces the wrong chunk, or where a chunking choice splits a fact away from the words that would find it. Explain the cause and the mitigation.

In [13]:
FAILURE_QUERY = 'What three time ranges can the analytics dashboard filter by?'
FAILURE_FACT = 'This Week, This Month, and This Year'
FAILURE_DOC = 'analytics'
FAILURE_CONFIG = 'small (120/20)'

cfg = configs[FAILURE_CONFIG]
relevant_chunk_indices = label_relevant(cfg['chunks'], cfg['sources'], FAILURE_DOC, FAILURE_FACT)
retrieved_chunk_indices = retrieve_indices(indices[FAILURE_CONFIG], FAILURE_QUERY, TOP_K)

print(f'Query: "{FAILURE_QUERY}"')
print(f'Chunking config: {FAILURE_CONFIG}')
print()
print('The chunk that actually answers this question:')
for i in relevant_chunk_indices:
    print(f'  [chunk {i}]', repr(cfg['chunks'][i]))
print()
print(f'The top-{TOP_K} chunks retrieval actually returned instead:')
for i in retrieved_chunk_indices:
    flag = '(this is the correct chunk)' if i in relevant_chunk_indices else '(wrong chunk)'
    print(f'  [chunk {i}] {flag}', repr(cfg['chunks'][i][:150]))

# How far down the ranking was the correct chunk -- close but bumped out of
# top-3, or nowhere near the top at all? These are very different stories.
all_chunks_ranked_by_similarity = retrieve_indices(indices[FAILURE_CONFIG], FAILURE_QUERY, k=len(cfg['chunks']))
rank_of_correct_chunk = [all_chunks_ranked_by_similarity.index(i) + 1 for i in relevant_chunk_indices]
print()
print(f'Rank of the correct chunk among all {len(cfg["chunks"])} small chunks: {rank_of_correct_chunk}')

# Does the same query succeed under the other two configs? (It should, per
# Part 2's table -- this confirms it's a chunking-config effect, not the
# query or the fact being unanswerable.)
print()
print('Compare: same query under the other two chunking configs')
for cfg_name in ['large (320/40)', 'by-paragraph']:
    other_cfg = configs[cfg_name]
    other_relevant = label_relevant(other_cfg['chunks'], other_cfg['sources'], FAILURE_DOC, FAILURE_FACT)
    other_retrieved = retrieve_indices(indices[cfg_name], FAILURE_QUERY, TOP_K)
    correct_chunk_was_retrieved = any(i in other_relevant for i in other_retrieved)
    print(f'  {cfg_name:16s} correct chunk retrieved: {correct_chunk_was_retrieved}')

Query: "What three time ranges can the analytics dashboard filter by?"
Chunking config: small (120/20)

The chunk that actually answers this question:
  [chunk 8] 'me window these figures cover, with three built-in options: This Week, This Month, and This Year, so an owner can check '

The top-3 chunks retrieval actually returned instead:
  [chunk 45] (wrong chunk) 'days, without having to change their regular weekly hours. All of this -- the weekly hours, buffer time, daily cap, and '
  [chunk 7] (wrong chunk) 'intments collection everything else in the app reads from. The dashboard lets an owner change the time window these figu'
  [chunk 69] (wrong chunk) 'e make it into the date and time pickers in the first place -- the filtering happens before the customer ever sees the c'

Rank of the correct chunk among all 127 small chunks: [8]

Compare: same query under the other two chunking configs
  large (320/40)   correct chunk retrieved: True
  by-paragraph     correct chunk retrieved: 

## Part 4: Retrieval Failure -- Wrong Chunk Retrieved

**Query:** "What three time ranges can the analytics dashboard filter by?"
**Config:** small (120/20)
**Fact:** "This Week, This Month, and This Year" (`analytics` doc)

### What happened

The chunk that actually answers the question is in chunk 8, `"...with three built-in options: This Week, This Month, and This Year, so an owner can check..."` -- but it wasn't retrieved. In its place, retrieval returned three wrong chunks: chunk 7, chunk 45, and chunk 69.

Chunk 7 is the closest.  It's the *immediately preceding* overlapping chunk, ending `"...time window these figu"` right where chunk 8 picks up `"me window these figures cover..."`. They're two overlapping slices of the same sentence, and the one without the answer outranked the one with it. The other two wrong chunks aren't even from the `analytics` doc -- chunk 45 and chunk 69 got pulled in on generic time-related vocabulary overlap with the query's "time ranges."

Checking the correct chunk's rank among all 127 small chunks: 8th. Not deeply buried, but comfortably outside the top-3 `k` used here.

### Cause

The problem is the short chunk size.  At 120 characters a chunk doesn't carry enough surrounding text to anchor it to "analytics dashboard filtering" specifically, so it ends up competing against its own overlapping neighbor and against unrelated chunks that happen to mention time. This isn't a case of the fact getting split across a boundary, it's fully there in chunk 8.  However the correct chunk on its own looks as generic as everything else nearby.

### Mitigation

Larger context per chunk fixes this directly, and it's already proven by the same experiment rather than assumed: both `large (320/40)` and `by-paragraph` retrieved the correct chunk successfully for this exact query. Their version of that chunk includes enough of the surrounding paragraph -- `AnalyticsSection`, "revenue charts," "computed client-side" -- to clearly anchor it to the analytics topic instead of competing on the word "time" alone.

Another option would be to raise K to 10 for smaller chunk sizes.  A low value for top K plus a small chunk size does not provide enough context to return a reliable answer.